# 07 - Validation Analysis

Three validation experiments using existing data (no LLM calls):

- **7A: Grade Correlation** — Do students with more weak skills get lower grades? (all 372 students)
- **7B: Relevance Analysis** — Is the LLM identifying skills relevant to the problem? (from experiment 05)
- **7C: Random Baseline** — Are our results better than random chance? (simulation)

In [ ]:
import json
import random
import ast
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / 'lib').exists() and (ROOT.parent / 'lib').exists():
    ROOT = ROOT.parent

import sys
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from lib.experiment_utils import load_best_attempts_df
from lib.mental_model import load_skill_map, calculate_student_profile, get_weak_skills

# Load base data
best_attempts_df = load_best_attempts_df()
skill_map, all_skills = load_skill_map()
subject_df = pd.read_csv(ROOT / 'dataset' / 'CodeWorkout' / 'LinkTables' / 'Subject.csv')

# Load experiment 05 results
RESULTS_PATH = ROOT / 'results' / '05_mental_model_comparison' / 'v1_simple_average'

exp05_results = {}
for csv_file in RESULTS_PATH.glob('student_*.csv'):
    df = pd.read_csv(csv_file)
    sid = int(df['SubjectID'].iloc[0])
    exp05_results[sid] = df
    print(f"Loaded student {sid}: {len(df)} rows")

print(f"\nTotal students in dataset: {best_attempts_df['SubjectID'].nunique()}")
print(f"Total students with grades: {len(subject_df)}")
print(f"Experiment 05 students loaded: {list(exp05_results.keys())}")

Main table: 201,570 rows
CodeState table: 69,627 rows
Subject table: 381 rows
Joined dataset: 191,584 rows
Best attempts: 15,375 rows (372 students, 50 problems)


FileNotFoundError: [Errno 2] No such file or directory: '/mnt/d/Projects/kintsugi/dataset/CodeWorkout/Subject.csv'

## 7A: Grade Correlation

**Question:** Do students with more weak skills actually get lower course grades?

This validates that our mental model (skill mastery calculation + weak skill threshold)
is measuring something real about student ability.

We compute the weak skill count for ALL 372 students and correlate with X-Grade.

In [ ]:
student_features = []

for sid in best_attempts_df['SubjectID'].unique():
    # Compute profile and weak skills
    profile = calculate_student_profile(sid, best_attempts_df, skill_map, all_skills)
    weak = get_weak_skills(profile, threshold=0.6)
    num_weak = len(weak)

    # Average problem score
    student_sub = best_attempts_df[best_attempts_df['SubjectID'] == sid]
    avg_score = student_sub['Score'].mean()
    num_problems = len(student_sub)

    # Get X-Grade from Subject.csv
    grade_row = subject_df[subject_df['SubjectID'] == sid]
    if grade_row.empty:
        continue
    x_grade = float(grade_row['X-Grade'].iloc[0])

    student_features.append({
        'SubjectID': sid,
        'AvgScore': round(avg_score, 4),
        'NumProblems': num_problems,
        'NumWeakSkills': num_weak,
        'XGrade': x_grade,
    })

features_df = pd.DataFrame(student_features)
print(f"Students with grade data: {len(features_df)}")
print(f"Weak skills range: {features_df['NumWeakSkills'].min()} - {features_df['NumWeakSkills'].max()}")
print(f"X-Grade range: {features_df['XGrade'].min():.2f} - {features_df['XGrade'].max():.2f}")
display(features_df.describe())

In [ ]:
# Correlation 1: Weak skills count vs X-Grade (our main validation)
r1, p1 = stats.pearsonr(features_df['NumWeakSkills'], features_df['XGrade'])

# Correlation 2: Average score vs X-Grade (sanity check — should be strong)
r2, p2 = stats.pearsonr(features_df['AvgScore'], features_df['XGrade'])

# Correlation 3: Weak skills vs Average score
r3, p3 = stats.pearsonr(features_df['NumWeakSkills'], features_df['AvgScore'])

print("=== Grade Correlation Results ===\n")
print(f"1. NumWeakSkills vs X-Grade:")
print(f"   Pearson r = {r1:.4f}, p-value = {p1:.6f}")
print(f"   {'Significant' if p1 < 0.05 else 'Not significant'} at p<0.05")
print(f"   {'Strong' if abs(r1) > 0.5 else 'Moderate' if abs(r1) > 0.3 else 'Weak'} correlation")

print(f"\n2. AvgScore vs X-Grade (sanity check):")
print(f"   Pearson r = {r2:.4f}, p-value = {p2:.6f}")

print(f"\n3. NumWeakSkills vs AvgScore:")
print(f"   Pearson r = {r3:.4f}, p-value = {p3:.6f}")

# Scatter plots
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].scatter(features_df['NumWeakSkills'], features_df['XGrade'], alpha=0.4, s=20)
z = np.polyfit(features_df['NumWeakSkills'], features_df['XGrade'], 1)
p = np.poly1d(z)
x_range = np.linspace(features_df['NumWeakSkills'].min(), features_df['NumWeakSkills'].max(), 100)
axes[0].plot(x_range, p(x_range), "r--", alpha=0.8)
axes[0].set_xlabel('Number of Weak Skills')
axes[0].set_ylabel('X-Grade')
axes[0].set_title(f'Weak Skills vs Grade (r={r1:.3f}, p={p1:.4f})')

axes[1].scatter(features_df['AvgScore'], features_df['XGrade'], alpha=0.4, s=20)
z2 = np.polyfit(features_df['AvgScore'], features_df['XGrade'], 1)
p2_line = np.poly1d(z2)
x_range2 = np.linspace(features_df['AvgScore'].min(), features_df['AvgScore'].max(), 100)
axes[1].plot(x_range2, p2_line(x_range2), "r--", alpha=0.8)
axes[1].set_xlabel('Average Problem Score')
axes[1].set_ylabel('X-Grade')
axes[1].set_title(f'Avg Score vs Grade (r={r2:.3f})')

axes[2].scatter(features_df['NumWeakSkills'], features_df['AvgScore'], alpha=0.4, s=20)
z3 = np.polyfit(features_df['NumWeakSkills'], features_df['AvgScore'], 1)
p3_line = np.poly1d(z3)
x_range3 = np.linspace(features_df['NumWeakSkills'].min(), features_df['NumWeakSkills'].max(), 100)
axes[2].plot(x_range3, p3_line(x_range3), "r--", alpha=0.8)
axes[2].set_xlabel('Number of Weak Skills')
axes[2].set_ylabel('Average Problem Score')
axes[2].set_title(f'Weak Skills vs Avg Score (r={r3:.3f})')

plt.tight_layout()

results_dir = ROOT / 'results' / '07_validation_analysis'
results_dir.mkdir(parents=True, exist_ok=True)
plt.savefig(str(results_dir / 'grade_correlation_plots.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f"\nPlot saved to: {results_dir / 'grade_correlation_plots.png'}")

## 7B: Relevance Analysis

**Question:** When the LLM identifies KC gaps, are those KCs actually relevant
to what the problem tests?

We already have Baseline_Relevance_F1 and Enriched_Relevance_F1 in our experiment 05
results. This section summarizes them.

In [ ]:
print("=== Relevance F1 Analysis ===\n")
print("Relevance F1 measures: are the LLM's identified gaps relevant to the problem's KCs?")
print("(Not about weak skills — just about whether the LLM is talking about the right topic)\n")

student_names = {14359: "Average", 10155: "Struggling", 14475: "High Performer"}

print(f"{'Student':<10} {'Type':<16} {'Problems':>8} {'Baseline':>10} {'Enriched':>10} {'Delta':>8}")
print("-" * 65)

all_failing = []

for sid in sorted(exp05_results.keys()):
    df = exp05_results[sid]
    fail_df = df[df['IsPerfect'] == False]
    label = student_names.get(sid, "Unknown")

    if len(fail_df) == 0:
        print(f"{sid:<10} {label:<16} {0:>8} {'N/A':>10} {'N/A':>10} {'N/A':>8}")
        continue

    b_rel = fail_df['Baseline_Relevance_F1'].mean()
    e_rel = fail_df['Enriched_Relevance_F1'].mean()
    delta = e_rel - b_rel

    print(f"{sid:<10} {label:<16} {len(fail_df):>8} {b_rel:>10.3f} {e_rel:>10.3f} {delta:>+8.3f}")
    all_failing.append(fail_df)

# Overall
if all_failing:
    combined = pd.concat(all_failing)
    b_overall = combined['Baseline_Relevance_F1'].mean()
    e_overall = combined['Enriched_Relevance_F1'].mean()
    print(f"\n{'Overall':<10} {'':<16} {len(combined):>8} {b_overall:>10.3f} {e_overall:>10.3f} {e_overall-b_overall:>+8.3f}")

# Per-problem detail for each student
for sid in sorted(exp05_results.keys()):
    df = exp05_results[sid]
    fail_df = df[df['IsPerfect'] == False]
    if len(fail_df) == 0:
        continue

    label = student_names.get(sid, "Unknown")
    print(f"\n--- Student {sid} ({label}) ---")

    for _, r in fail_df.iterrows():
        # Parse KC tags (handle both string and list formats)
        b_tags = r['Baseline_KCTags']
        e_tags = r['Enriched_KCTags']
        if isinstance(b_tags, str):
            try:
                b_tags = ast.literal_eval(b_tags)
            except:
                b_tags = []
        if isinstance(e_tags, str):
            try:
                e_tags = ast.literal_eval(e_tags)
            except:
                e_tags = []

        b_tag_str = ", ".join(b_tags) if b_tags else "(none)"
        e_tag_str = ", ".join(e_tags) if e_tags else "(none)"

        print(f"  Problem {int(r['ProblemID'])} ({r['ScorePct']:.1f}%): "
              f"Baseline Rel={r['Baseline_Relevance_F1']:.3f}, "
              f"Enriched Rel={r['Enriched_Relevance_F1']:.3f}")

## 7C: Random Baseline Comparison

**Question:** If we randomly assigned KC tags instead of using the LLM,
would we get similar WeakSkillOverlap F1 scores?

We simulate 1000 random tag assignments per problem and compare against
our actual results.

In [ ]:
EXACT_KC_TAGS = [
    'If/Else', 'NestedIf', 'While', 'For', 'NestedFor',
    'Math+-*/', 'Math%', 'LogicAndNotOr', 'LogicCompareNum', 'LogicBoolean',
    'StringFormat', 'StringConcat', 'StringIndex', 'StringLen',
    'StringEqual', 'CharEqual', 'ArrayIndex', 'DefFunction'
]

NUM_SIMULATIONS = 1000
random.seed(42)

print("=== Random Baseline Comparison ===\n")
print(f"Simulating {NUM_SIMULATIONS} random tag assignments per problem")
print(f"KC tag pool: {len(EXACT_KC_TAGS)} tags\n")

random_results = []

for sid in sorted(exp05_results.keys()):
    df = exp05_results[sid]
    fail_df = df[df['IsPerfect'] == False]
    label = student_names.get(sid, "Unknown")

    if len(fail_df) == 0:
        print(f"Student {sid} ({label}): No failing problems, skipping.\n")
        random_results.append({
            'SubjectID': sid, 'Label': label, 'NumWeakSkills': 0,
            'Random_F1_Mean': 0.0, 'Random_F1_Std': 0.0, 'Random_F1_95th': 0.0,
            'Actual_Baseline_F1': 0.0, 'Actual_Enriched_F1': 0.0,
            'Baseline_vs_Random': 'N/A', 'Enriched_vs_Random': 'N/A',
        })
        continue

    # Get weak skills for this student
    weak_col = fail_df['WeakSkills'].iloc[0]
    if isinstance(weak_col, str):
        try:
            weak_list = ast.literal_eval(weak_col)
        except:
            weak_list = []
    else:
        weak_list = weak_col if isinstance(weak_col, list) else []

    weak_set = set(weak_list)

    # For each simulation, randomly assign tags to each problem and compute avg F1
    sim_f1s = []
    for _ in range(NUM_SIMULATIONS):
        problem_f1s = []
        for _, row in fail_df.iterrows():
            # How many tags did the LLM actually produce? Use baseline count
            b_tags = row['Baseline_KCTags']
            if isinstance(b_tags, str):
                try:
                    b_tags = ast.literal_eval(b_tags)
                except:
                    b_tags = []
            tag_count = len(b_tags) if b_tags else 3

            # Pick random tags
            random_tags = set(random.sample(EXACT_KC_TAGS, min(tag_count, len(EXACT_KC_TAGS))))

            # Calculate F1 against weak skills
            if not random_tags or not weak_set:
                problem_f1s.append(0.0)
                continue
            overlap = random_tags & weak_set
            precision = len(overlap) / len(random_tags)
            recall = len(overlap) / len(weak_set)
            f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0
            problem_f1s.append(f1)

        sim_f1s.append(np.mean(problem_f1s))

    random_mean = np.mean(sim_f1s)
    random_std = np.std(sim_f1s)
    random_95th = np.percentile(sim_f1s, 95)

    actual_baseline = fail_df['Baseline_WeakOverlap_F1'].mean()
    actual_enriched = fail_df['Enriched_WeakOverlap_F1'].mean()

    b_vs_random = "ABOVE" if actual_baseline > random_95th else ("NEAR" if actual_baseline > random_mean else "BELOW")
    e_vs_random = "ABOVE" if actual_enriched > random_95th else ("NEAR" if actual_enriched > random_mean else "BELOW")

    print(f"Student {sid} ({label}, {len(weak_list)} weak skills):")
    print(f"  Random F1: {random_mean:.3f} (+/- {random_std:.3f}), 95th percentile: {random_95th:.3f}")
    print(f"  Actual Baseline F1:  {actual_baseline:.3f}  [{b_vs_random} random]")
    print(f"  Actual Enriched F1:  {actual_enriched:.3f}  [{e_vs_random} random]")
    print(f"  Enriched vs Random mean: {actual_enriched - random_mean:+.3f}\n")

    random_results.append({
        'SubjectID': sid, 'Label': label, 'NumWeakSkills': len(weak_list),
        'Random_F1_Mean': round(random_mean, 4), 'Random_F1_Std': round(random_std, 4),
        'Random_F1_95th': round(random_95th, 4),
        'Actual_Baseline_F1': round(actual_baseline, 4),
        'Actual_Enriched_F1': round(actual_enriched, 4),
        'Baseline_vs_Random': b_vs_random, 'Enriched_vs_Random': e_vs_random,
    })

random_df = pd.DataFrame(random_results)
display(random_df)

In [ ]:
results_dir = ROOT / 'results' / '07_validation_analysis'
results_dir.mkdir(parents=True, exist_ok=True)

# Save grade correlation data
features_df.to_csv(results_dir / 'grade_correlation_data.csv', index=False)

# Save random baseline results
random_df.to_csv(results_dir / 'random_baseline_results.csv', index=False)

# Save metadata
metadata = {
    "experiment": "07_validation_analysis",
    "run_timestamp": datetime.now().isoformat(),
    "sections": {
        "7A_grade_correlation": {
            "students_analyzed": len(features_df),
            "pearson_r_weakskills_vs_grade": round(float(r1), 4),
            "p_value": round(float(p1), 6),
            "significant": bool(p1 < 0.05),
        },
        "7B_relevance_analysis": {
            "students": list(exp05_results.keys()),
            "overall_baseline_relevance_f1": round(float(b_overall), 4) if all_failing else None,
            "overall_enriched_relevance_f1": round(float(e_overall), 4) if all_failing else None,
        },
        "7C_random_baseline": {
            "num_simulations": NUM_SIMULATIONS,
            "results": random_results,
        }
    }
}

with open(results_dir / 'metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2, default=str)

print(f"All results saved to: {results_dir}")
print(f"  grade_correlation_data.csv")
print(f"  grade_correlation_plots.png")
print(f"  random_baseline_results.csv")
print(f"  metadata.json")